In [1]:
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification, CLIPModel, CLIPProcessor
from pathlib import Path
from tqdm import tqdm
import os

# --- 1. 설정 (Configuration) ---

# [사용자 설정] 파일 경로들을 지정합니다.
# DEV_TEST_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv"
DEV_TEST_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/preprocess/caption/test_preprocessed_for_clip.csv"
SAMPLE_SUBMISSION_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/sample_submission.csv"
BASE_IMAGE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images"
# 최종적으로 저장될 제출 파일 이름
FINAL_SUBMISSION_PATH = "clip_with_llm_2.csv"

# 디바이스 설정 (GPU가 있으면 자동으로 사용)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {DEVICE}")

# --- 2. 모델 및 프로세서 로드 ---
print("CLIP 모델과 프로세서를 로드하는 중...")
MODEL_NAME = "openai/clip-vit-base-patch32" # 이 변수는 사용되지 않으므로 무시해도 됩니다.
model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

# ### 핵심 수정 부분 ###
# 로드한 모델을 지정된 디바이스(GPU 또는 CPU)로 보냅니다.
model.to(DEVICE)

model.eval() # 모델을 추론 모드로 설정
print("모델 로드 완료.")

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


사용 디바이스: cuda
CLIP 모델과 프로세서를 로드하는 중...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


모델 로드 완료.


In [2]:
# 전체 파라미터와 학습 가능한 파라미터 수를 계산
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total_params:,}개 ({total_params / 1_000_000:.2f}M)")
print(f"학습 가능한 파라미터: {trainable_params:,}개 ({trainable_params / 1_000_000:.2f}M)")

전체 파라미터: 427,616,513개 (427.62M)
학습 가능한 파라미터: 427,616,513개 (427.62M)


In [3]:


# --- 3. 데이터 로드 및 추론 루프 ---

# 추론할 이미지와 질문 정보가 담긴 dev_test.csv 로드
try:
    df_test = pd.read_csv(DEV_TEST_CSV_PATH)
    print(f"총 {len(df_test)}개의 데이터에 대한 추론을 시작합니다.")
except FileNotFoundError:
    print(f"오류: {DEV_TEST_CSV_PATH} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# 결과를 채워넣을 sample_submission.csv 로드
try:
    df_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
except FileNotFoundError:
    print(f"오류: {SAMPLE_SUBMISSION_PATH} 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()

# 예측된 인덱스(0,1,2,3)를 정답 레이블(A,B,C,D)로 변환하기 위한 딕셔너리
index_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}

# tqdm을 사용하여 진행 상황을 시각적으로 표시
for index, row in tqdm(df_test.iterrows(), total=df_test.shape[0], desc="추론 진행률"):

    test_id = row['ID']
    relative_img_path = row['img_path']
    choices = [row['caption_A'], row['caption_B'], row['caption_C'], row['caption_D']]
    image_filename = os.path.basename(relative_img_path)
    full_image_path = Path(BASE_IMAGE_DIR) / image_filename

    try:
        image = Image.open(full_image_path).convert("RGB")
        # 입력 데이터를 GPU로 보내기 전에 먼저 텐서로 변환합니다.
        inputs = processor(images=image, text=choices, return_tensors="pt", padding=True)
        # 딕셔너리의 각 텐서를 DEVICE로 보냅니다.
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            # 모델과 입력 데이터가 모두 같은 DEVICE에 있으므로 정상적으로 연산됩니다.
            outputs = model(**inputs)

        logits_per_image = outputs.logits_per_image.cpu() # 계산 후에는 CPU로 가져와서 처리하는 것이 안전합니다.

        # 가장 높은 점수를 가진 선택지의 '인덱스' 찾기
        predicted_index = logits_per_image.argmax(-1).item()
        # 인덱스를 'A', 'B', 'C', 'D' 문자로 변환
        prediction_label = index_to_label[predicted_index]

        # df_submission에서 현재 ID와 일치하는 행을 찾아 'answer' 컬럼을 A,B,C,D로 업데이트
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = prediction_label

    except FileNotFoundError:
        print(f"\n경고: ID '{test_id}'의 이미지 파일을 찾을 수 없습니다. 경로: {full_image_path}")
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = 'Error_FileNotFound' # 에러 원인을 명시적으로 표시
    except Exception as e:
        print(f"\n오류: ID '{test_id}' 처리 중 예외 발생: {e}")
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = 'Error_Exception'

# --- 4. 결과 저장 및 출력 ---

df_submission.to_csv(FINAL_SUBMISSION_PATH, index=False)

print("\n" + "="*50)
print("추론이 완료되었습니다.")
print(f"결과가 {FINAL_SUBMISSION_PATH} 파일에 저장되었습니다.")
print("\n최종 제출 파일 미리보기 (상위 5개):")
print(df_submission.head())
print("="*50)

총 60개의 데이터에 대한 추론을 시작합니다.


추론 진행률: 100%|██████████| 60/60 [00:02<00:00, 25.65it/s]


추론이 완료되었습니다.
결과가 clip_with_llm_2.csv 파일에 저장되었습니다.

최종 제출 파일 미리보기 (상위 5개):
         ID answer
0  TEST_000      A
1  TEST_001      B
2  TEST_002      C
3  TEST_003      C
4  TEST_004      B
